In [14]:
# Problem Statement  
# Cryptocurrency markets are highly volatile, and liquidity plays a crucial role in market stability. Liquidity refers to 
#the ease with which assets can be bought or sold without significantly impacting the price. A lack of liquidity 
#can lead to increased price fluctuations and market instability.
 #In this project, you are required to build a machine learning model to predict cryptocurrency liquidity levels 
#based on various market factors such as trading volume, transaction patterns, exchange listings, and social 
#media activity. The objective is to detect liquidity crises early to help traders and exchange platforms 
#manage risks effectively.
 #Your final model should provide insights into market stability by forecasting liquidity variations, allowing 
#traders and financial institutions to make informed decisions
import os
import numpy as np
import pandas as pd
from glob import glob
from datetime import timedelta
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import xgboost as xgb
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

# ----- Helper functions -----

def load_data_from_folder(folder='data'):
    """Load all CSV files in data folder and return concatenated DataFrame.
    Assumes each CSV has at least: timestamp (or date), open, high, low, close, volume"""
    files = glob(os.path.join(folder, '*.csv'))
    if not files:
        raise FileNotFoundError(f'No CSV files found in {folder}. Place dataset CSV(s) there.')
    dfs = []
    for f in files:
        df = pd.read_csv(f)
        # try common date columns
        for col in ['timestamp', 'date', 'time']:
            if col in df.columns:
                df['date'] = pd.to_datetime(df[col])
                break
        if 'date' not in df.columns:
            # try index as date
            try:
                df['date'] = pd.to_datetime(df.iloc[:,0])
            except Exception:
                raise ValueError(f'Could not find a date column in {f}.')
        dfs.append(df)
    data = pd.concat(dfs, ignore_index=True)
    data = data.sort_values('date').reset_index(drop=True)
    return data


def compute_liquidity_measures(df, price_col='close', volume_col='volume', window=24):
    """Compute several liquidity proxy features."""
    df = df.copy()
    df['return'] = df[price_col].pct_change().fillna(0)
    df['abs_return'] = df['return'].abs()
    df['rolling_vol'] = df['return'].rolling(window=window, min_periods=1).std()
    df['rolling_volume'] = df[volume_col].rolling(window=window, min_periods=1).sum()
    df['amihud'] = (df['abs_return'] / (df[volume_col].replace(0, np.nan))).rolling(window=window, min_periods=1).mean().fillna(0)
    if 'high' in df.columns and 'low' in df.columns:
        df['rel_spread'] = ((df['high'] - df['low']) / df[price_col]).rolling(window=window, min_periods=1).mean()
    else:
        df['rel_spread'] = 0
    df['liq_index_raw'] = df['rolling_volume'] / (df['rolling_vol'].replace(0, np.nan))
    df['liq_index_raw'] = df['liq_index_raw'].replace([np.inf, -np.inf], np.nan).fillna(0)
    return df


def create_labels(df, target_col='liq_index_raw', strategy='quantile', low_q=0.33, high_q=0.67):
    """Create categorical liquidity labels based on quantiles."""
    df = df.copy()
    if strategy == 'quantile':
        low = df[target_col].quantile(low_q)
        high = df[target_col].quantile(high_q)
        def label(x):
            if x <= low:
                return 0
            elif x <= high:
                return 1
            else:
                return 2
        df['liq_label'] = df[target_col].apply(label)
    else:
        raise NotImplementedError('Only quantile strategy implemented')
    return df

# ----- Main flow -----

try:
    df = load_data_from_folder('data')
except Exception as e:
    print('*** DATA LOADING ERROR: ***')
    print(e)
    print('\nMake sure you downloaded CSV(s) to the folder `data/` and they contain timestamp/date and price/volume columns.')
    raise

print('Data loaded. Sample:')
print(df.head())

if df['date'].duplicated().any():
    df = df.drop_duplicates(subset=['date']).reset_index(drop=True)

for col in ['close', 'Close', 'price']:
    if col in df.columns and 'close' not in df.columns:
        df.rename(columns={col: 'close'}, inplace=True)
for col in ['volume', 'Volume', 'vol']:
    if col in df.columns and 'volume' not in df.columns:
        df.rename(columns={col: 'volume'}, inplace=True)

for c in ['close','volume']:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors='coerce')

df = compute_liquidity_measures(df, price_col='close', volume_col='volume', window=24)
df = create_labels(df, target_col='liq_index_raw', strategy='quantile', low_q=0.33, high_q=0.67)

feature_cols = ['close', 'volume', 'rolling_volume', 'rolling_vol', 'amihud', 'rel_spread']
feature_cols = [c for c in feature_cols if c in df.columns]
X = df[feature_cols].fillna(0).copy()
Y = df['liq_label'].copy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

split_idx = int(len(df) * 0.8)
X_train, X_test = X_scaled[:split_idx], X_scaled[split_idx:]
y_train, y_test = Y[:split_idx], Y[split_idx:]

rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
print('\nClassification report for RandomForest:')
print(classification_report(y_test, y_pred, target_names=['Low','Medium','High']))

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=['Low','Medium','High'], yticklabels=['Low','Medium','High'])
plt.title('Confusion Matrix - RandomForest')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.show()

xgb_clf = xgb.XGBClassifier(use_label_encoder=False, eval_metric='mlogloss', n_jobs=4, random_state=42)
xgb_clf.fit(X_train, y_train)
y_pred_xgb = xgb_clf.predict(X_test)
print('\nClassification report for XGBoost:')
print(classification_report(y_test, y_pred_xgb, target_names=['Low','Medium','High']))

try:
    fi = xgb_clf.feature_importances_
    for col, imp in zip(feature_cols, fi):
        print(f'{col}: {imp:.4f}')
except Exception:
    pass

os.makedirs('models', exist_ok=True)
joblib.dump(rf, 'models/rf_liquidity.pkl')
joblib.dump(xgb_clf, 'models/xgb_liquidity.pkl')
joblib.dump(scaler, 'models/scaler.pkl')
print('\nModels saved to models/ folder')

def predict_liquidity(df_new, model_path='models/xgb_liquidity.pkl'):
    """Given a new dataframe with same columns, return predicted liquidity label and probabilities."""
    m = joblib.load(model_path)
    sc = joblib.load('models/scaler.pkl')
    df_feat = compute_liquidity_measures(df_new, price_col='close', volume_col='volume', window=24)
    Xf = df_feat[feature_cols].fillna(0)
    Xs = sc.transform(Xf)
    preds = m.predict(Xs)
    probs = m.predict_proba(Xs)
    df_feat['pred_label'] = preds
    df_feat['pred_conf'] = probs.max(axis=1)
    return df_feat

print('\nNotebook run complete. Check models/ for saved models and adjust feature engineering as necessary.')


Data loaded. Sample:
             coin  symbol         price     1h    24h     7d    24h_volume  \
0         Bitcoin     BTC  4.085946e+04  0.022  0.030  0.055  3.539076e+10   
1  Iron Bank EURO   IBEUR  1.080000e+00  0.000 -0.004  0.009  9.525810e+04   
2       Prometeus    PROM  7.960000e+00  0.017  0.008  0.015  1.069360e+06   
3    MaidSafeCoin    MAID  2.949200e-01  0.023  0.010  0.045  3.041720e+03   
4    Bezoge Earth  BEZOGE  3.051000e-09  0.012 -0.005 -0.041  1.894020e+05   

        mkt_cap       date  
0  7.709915e+11 2022-03-16  
1  1.300442e+08 2022-03-16  
2  1.302007e+08 2022-03-16  
3  1.327759e+08 2022-03-16  
4  1.329136e+08 2022-03-16  


KeyError: 'volume'